In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
plotly.offline.init_notebook_mode()
display(HTML(
    '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
))

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)
plt.close('all')

# Load Data

In [ ]:
multiRun_dir = f"{home_dir}/CHARLES/multiRuns/"
plotFolder = f"{multiRun_dir}"

roomVentilationMI = pd.read_csv(f"{multiRun_dir}/roomVentilationMIEmulation.csv", index_col = [0,1])
flowStatsMI = pd.read_csv(f"{multiRun_dir}/flowStatsMIEmulation.csv", index_col=[0,1])

In [ ]:
# Train/Dev/Test Assignment
# Create a column called split and assign 70 % to train, 10% to dev, and 20% to test in roomVentilationMI
random.seed(42)  # For reproducibility
roomVentilationMI["split"] = roomVentilationMI.index.to_series().apply(lambda _: random.random())
roomVentilationMI["split"] = roomVentilationMI["split"].apply(lambda x: "train" if x < 0.7 else ("dev" if x < 0.8 else "test"))

for (run, room), row in roomVentilationMI.iterrows():
    windowKeyCols = roomVentilationMI.columns[
        roomVentilationMI.columns.str.contains("windowKeys")
    ].tolist()
    windowKeys = row[windowKeyCols].dropna()
    for windowKey in windowKeys:
        flowStatsMI.loc[(run, windowKey), "split"] = row["split"]

In [ ]:
df = flowStatsMI.copy()
# df = df[df["slAll"] == False]
# normalize x_cols. Flow quantities to be normalized by WS. Pressures to be normalized by W**2:

# df['p-noInt_optp0-q_modelC_d'] = df['p-noInt_optp0-q_model'] * df['p-noInt_optp0-C_d']
# for col in df.columns:
#     if "mag" in col or "shear" in col or "normal" in col:
#         df[col] = df[col] / df['p-noInt_optp0-q_modelC_d']

df["skylight"] = df['openingType'].apply(lambda x: 1 if "skylight" in x else 0)
df["cross"] = df['openingType'].apply(lambda x: 1 if "cross" in x else 0)
df["single"] = df['openingType'].apply(lambda x: 1 if "single" in x else 0)
df["dual"] = df['openingType'].apply(lambda x: 1 if "dual" in x else 0)
df["corner"] = df['openingType'].apply(lambda x: 1 if "corner" in x else 0)
Sdelp = np.sign(df['p-noInt_optp0-q_model'])
Sdelp[Sdelp == 0] = 1  # Assign 1 to zero values
df["Sdelp"] = Sdelp
df["EP_shear-noInt-qIn"] = df["EP_shear-noInt"] * df["Sdelp"] > 0
df["EP_shear-noInt-qOut"] = df["EP_shear-noInt"] * df["Sdelp"] <= 0
df["EP_shear_o_qmodel"] = df["EP_shear-noInt"] / df["p-noInt_optp0-q_model"]
df["rev-mass_flux"] =df["net-mass_flux"] -  df["mean-mass_flux"].abs()
df["rev-mass_flux-Norm"] = df["rev-mass_flux"] / df["WS"]
df["p_intensity-noInt"] = df["p_rms-noInt"] / df["p-noInt_optp0-q_model"]**2

p_norm_cols = []
u_norm_cols = []
no_norm_cols = []
for col in df.columns:
    if ("noInt" in col or col == "flux" or "EP" in col) and "Norm" not in col:
        if "-p0" in col or "p_" in col or "(p)" in col or "p0meas" in col or "u**2" in col:
            p_norm_cols.append(col)
        elif "mag" in col or "shear" in col or "normal" in col or "flux" in col or "(u" in col or "q_model" in col:
            u_norm_cols.append(col)
        else:
            no_norm_cols.append(col)

print(f"Normalizing p cols: {sorted(p_norm_cols)}")
print(f"Normalizing u cols: {sorted(u_norm_cols)}")
print(f"Not normalizing cols: {sorted(no_norm_cols)}")

# Normalize pressure columns by WS^2
for col in p_norm_cols:
    df[f"{col}-Norm"] = df[col].div(df["WS"]**2, axis=0)
# Normalize velocity columns by WS
for col in u_norm_cols:
    df[f"{col}-Norm"] = df[col].div(df["WS"], axis=0)

In [ ]:
roomVentilationMI["flux-Norm"] = roomVentilationMI["flux"] / roomVentilationMI["WS"]
roomVentilationMI["p-noInt_optp0-q_model-Norm"] = roomVentilationMI["p-noInt_optp0-q_model"] / roomVentilationMI["WS"]

In [ ]:
grouped = df.groupby("WS").mean(numeric_only=True)
ratio = grouped.loc[4] / grouped.loc[2]  # Should be about 4x difference`
ratio = pd.DataFrame(ratio).reset_index()
ratio

In [ ]:
df = df[~((df["roomType"] == "single") & (df["slAll"] == False))]  # remove single rooms without skylights
roomVentilationMI = roomVentilationMI[~((roomVentilationMI["roomType"] == "single") & (roomVentilationMI["slAll"] == False))]  # remove single rooms without skylights

df["all"] = True

In [ ]:
y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
data = df.copy()

In [ ]:
def calculate_normalized_rmse(y, y_pred, normalization='std'):
    """
    Calculate RMSE and normalized RMSE for any regression model.
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, n_features)
        Input features
    y : array-like, shape (n_samples,)
        Target values
    normalization : str, optional (default='std')
        Method for normalization:
        - 'std': normalize by standard deviation of y
        - 'mean': normalize by mean of y
        - 'range': normalize by range of y
        
    Returns:
    --------
    rmse : float
        Root Mean Square Error
    nrmse : float
        Normalized Root Mean Square Error
    """
    import numpy as np
    
    # Calculate RMSE
    mse = np.mean((y - y_pred)**2)
    rmse = np.sqrt(mse)
    
    # Calculate normalized RMSE
    if normalization == 'mean':
        # Normalize by mean of observed values
        nrmse = rmse / np.mean(np.abs(y))
    elif normalization == 'range':
        # Normalize by range of observed values
        nrmse = rmse / (np.max(y) - np.min(y))
    else:  # default: 'std'
        # Normalize by standard deviation of observed values
        nrmse = rmse / np.std(y)
    
    return rmse, nrmse

In [ ]:
def plot_ventilation_model_fit(data, y_var, x_var, x_var2=None, hue="roomType", style="slAll",
                                model_func=pyafn.ventilationReDecomp_q, model_name="Ventilation Model",
                                p0=[1.0, 0.5, 1.0], bounds=([0.1, 0, 0], [np.inf, np.inf, 2]),
                                adjustData=False, show_assymptotes=False,
                                group_param2_means=None, group_param2_half_width=0.001,
                                axes=None, show_row_titles=True, set_axis_labels=True,
                                show_figure_legend=True, add_numeric_colorbar=False,
                                hue_norm=None, palette_numeric="viridis", figure_suptitle=True,
                                colorbar_rect=None, colorbar_orientation="horizontal", return_data=False,
                                return_params=False):
    """
    Plot ventilation model fits comparing modeled vs LES flux velocities.
    
    Parameters:
    -----------
    data : DataFrame
        Input data containing all required columns
    y_var : str
        Name of the dependent variable (LES flux)
    x_var : str
        Name of the primary independent variable (modeled flux)
    x_var2 : str, optional
        Name of secondary independent variable (e.g., turbulence intensity)
    hue : str
        Column name for color grouping in scatter plot
    style : str
        Column name for marker style in scatter plot
    model_func : callable
        Function to use for curve fitting
    model_name : str
        Name of the model for plot title
    p0 : list
        Initial parameter guesses for curve_fit
    bounds : tuple
        Parameter bounds for curve_fit
    adjustData : bool
        If True, adjust x_var based on fitted model; if False, plot regression lines
    show_assymptotes : bool
        If True, show asymptote lines in each subplot
    group_param2_means : dict, optional
        Optional mapping for second fit parameter initialization by subgroup, keyed by
        (skylight, Sdelp) where skylight is 0/1 (or False/True) and Sdelp is +/-1.
    group_param2_half_width : float, optional
        Half-width used to center the second parameter bounds around subgroup means
    axes : array-like of matplotlib axes, optional
        External axes for composition. Should contain two axes (Window, Skylight)
    show_row_titles : bool
        If True, title each row as Window/Skylight
    set_axis_labels : bool
        If True, set x/y labels for each subplot
    show_figure_legend : bool
        If True, add a figure-level legend for categorical hue/style
    add_numeric_colorbar : bool
        If True and hue is numeric, add a shared colorbar for these two subplots
    hue_norm : matplotlib.colors.Normalize, optional
        Normalization for numeric hue coloring
    palette_numeric : str
        Colormap for numeric hue
    figure_suptitle : bool
        If True, add a figure-level super title
    colorbar_orientation : str
        "horizontal" or "vertical" numeric colorbar orientation
    return_data : bool
        If True, return the (possibly adjusted) x_var series as a third return value
    return_params : bool
        If True, return a dict of fitted parameters keyed by (title, lbl), e.g.
        ("Window", "Flow Entering"), ("Window", "Flow Exiting"),
        ("Skylight", "Flow Entering"), ("Skylight", "Flow Exiting").
    
    Returns:
    --------
    fig, axs : matplotlib figure and axes objects
    """
    from scipy.optimize import curve_fit
    import seaborn as sns
    import numpy as np
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable

    data = data.copy()
    using_external_axes = axes is not None
    if using_external_axes:
        axs = np.array(axes).flatten()
        if len(axs) < 2:
            raise ValueError("When providing axes, pass at least two axes (Window, Skylight).")
        axs = axs[:2]
        fig = axs[0].figure
    else:
        n_rows = 1
        fig, axs = plt.subplots(n_rows, 2, figsize=(12, 6*n_rows), dpi=140, sharex=True, sharey=True)
        axs = np.array(axs).flatten()
    
    # Configure axes
    for i in range(len(axs)):
        axs[i].grid(True, alpha=0.3)
        axs[i].tick_params(labelsize=16)
        if set_axis_labels:
            axs[i].set_xlabel("Modeled Flux Velocity", fontsize=20)
            axs[i].set_ylabel("LES Flux Velocity", fontsize=20)
        axs[i].set_xlim(-0.6, 0.6)
        axs[i].set_ylim(-0.6, 0.6)
    
    hue_is_numeric = pd.api.types.is_numeric_dtype(data[hue])
    if hue_is_numeric and hue_norm is None:
        finite_vals = pd.to_numeric(data[hue], errors="coerce")
        finite_vals = finite_vals[np.isfinite(finite_vals)]
        if len(finite_vals) > 0:
            hue_norm = Normalize(vmin=finite_vals.min(), vmax=finite_vals.max())
    
    # Store asymptote legend info to re-apply after the removal loop
    asymptote_legend_handles = None
    asymptote_legend_labels = None
    scatter_collection_for_cbar = None
    fitted_params = {}

    # Plot for each skylight condition
    for i in range(2):
        sl_val = bool(i)
        plotdf = data[data["skylight"] == sl_val].copy()
        plotdf = plotdf.sort_values(by=[hue, style])
        
        min_val = min(plotdf[x_var].min(), plotdf[y_var].min())
        max_val = max(plotdf[x_var].max(), plotdf[y_var].max())
        
        # 1:1 reference line
        xs = np.array([min_val, max_val])
        axs[i].plot(xs, xs, 'r--', label='1:1 Line')
        
        # Title
        title = "Skylight" if sl_val else "Window"
        if show_row_titles:
            axs[i].set_title(title, fontsize=20)
        
        # Fit regression lines for each sign group (entering/exiting flow)
        for s, stl, lbl in [(1, '-', 'Flow Entering'), (-1, '-', 'Flow Exiting')]:
            regdf_cols = [x_var, y_var] + ([x_var2] if x_var2 else [])
            regdf = plotdf[plotdf["Sdelp"] == s][regdf_cols]
            regdf_abs = regdf.abs()
            regdf_abs = regdf_abs.sort_values(by=x_var)
            
            # Skip fit if subgroup has no samples
            if regdf_abs.empty:
                print(f"Skipping fit for {title}, {lbl}: no samples")
                continue
            
            # Build subgroup-specific fit initialization when requested
            fit_p0 = list(p0)
            fit_bounds = (list(bounds[0]), list(bounds[1]))
            if group_param2_means is not None and len(fit_p0) > 1:
                subgroup_keys = [
                    (int(sl_val), int(s)),
                    (bool(sl_val), int(s)),
                    (sl_val, s),
                ]
                subgroup_mean = None
                for key in subgroup_keys:
                    if key in group_param2_means:
                        subgroup_mean = group_param2_means[key]
                        break
                
                if subgroup_mean is not None and np.isfinite(subgroup_mean):
                    fit_p0[1] = float(subgroup_mean)
                    if len(fit_bounds[0]) > 1 and len(fit_bounds[1]) > 1:
                        fit_bounds[0][1] = float(subgroup_mean) - group_param2_half_width
                        fit_bounds[1][1] = float(subgroup_mean) + group_param2_half_width
                    print(
                        f"Init {title}, {lbl}: param2_mean={subgroup_mean:.4f}, "
                        f"bounds=({fit_bounds[0][1]:.4f}, {fit_bounds[1][1]:.4f})"
                    )
                else:
                    print(f"Init {title}, {lbl}: subgroup mean missing; using provided p0/bounds")
            
            # Fit model
            if x_var2:
                X_fit = (regdf_abs[x_var], regdf_abs[x_var2])
            else:
                X_fit = regdf_abs[x_var]
            
            popt = curve_fit(model_func, X_fit, regdf_abs[y_var], p0=fit_p0, bounds=fit_bounds)[0]
            print(f"Fitted parameters for {title}, {lbl}: popt={popt}")
            fitted_params[(title, lbl)] = popt
            
            mask = plotdf["Sdelp"] == s
            if x_var2:
                y_pred_model = model_func((np.abs(plotdf.loc[mask, x_var].values), np.abs(plotdf.loc[mask, x_var2].values)), *popt) * s
            else:
                y_pred_model = model_func(np.abs(plotdf.loc[mask, x_var].values), *popt) * s
            
            if adjustData:
                # Adjust x values based on fitted model
                plotdf.loc[mask, x_var] = y_pred_model
            else:
                # Plot regression line
                if x_var2:
                    y_pred = model_func((regdf_abs[x_var].values, regdf_abs[x_var2].values), *popt)
                else:
                    y_pred = model_func(regdf_abs[x_var], *popt)
                
                x_plot = regdf_abs[x_var] * s
                y_plot = y_pred * s
                
                fit_label = "Model" if s > 0 else None
                axs[i].plot(x_plot, y_plot, color="k", linestyle=stl, label=fit_label, linewidth=2.5)
                
                # Add asymptote plots if requested
                if show_assymptotes:
                    if x_var2:
                        y_pred_lower = s * model_func((regdf_abs[x_var].values, regdf_abs[x_var2].values), *popt, I_crit=0.001)
                        y_pred_upper = s * model_func((regdf_abs[x_var].values, regdf_abs[x_var2].values), *popt, I_crit=1000)
                    else:
                        y_pred_lower = s * model_func(regdf_abs[x_var], *popt, I_crit=0.001)
                        y_pred_upper = s * model_func(regdf_abs[x_var], *popt, I_crit=1000)
                    
                    upper_label = "Mean Dominated" if s > 0 else None
                    lower_label = "Fluctuation Dominated" if s > 0 else None
                    axs[i].plot(x_plot, y_pred_upper, color="blue", linestyle='--', label=upper_label, linewidth=1)
                    axs[i].plot(x_plot, y_pred_lower, color="green", linestyle='--', label=lower_label, linewidth=1)
            
            # Calculate model-vs-observed metrics
            y_obs_model = plotdf.loc[mask, y_var].values
            rmse, nrmse = calculate_normalized_rmse(
                y_obs_model,
                y_pred_model,
                normalization='std'
            )
            print(f"NRMSE: {nrmse:.2f}, RMSE: {rmse:.3f}")
            
            error = y_pred_model - y_obs_model
            print(f"Bias: {np.mean(error):.4f}, Error STD: {np.std(error):.4f}")

        
        # Scatter plot
        legend_mode = "auto"
        if hue_is_numeric:
            legend_mode = False
        scatter_ax = sns.scatterplot(
            data=plotdf, x=x_var, y=y_var,
            hue=hue, style=style,
            alpha=0.6, s=20, ax=axs[i],
            palette=palette_numeric if hue_is_numeric else "colorblind",
            hue_norm=hue_norm if hue_is_numeric else None,
            legend=legend_mode
        )
        if hue_is_numeric and len(scatter_ax.collections) > 0:
            scatter_collection_for_cbar = scatter_ax.collections[0]
            if scatter_collection_for_cbar is not None:
                scatter_collection_for_cbar.set_cmap(palette_numeric)
                if hue_norm is not None:
                    scatter_collection_for_cbar.set_norm(hue_norm)
        
        if adjustData:
            rmse, nrmse = calculate_normalized_rmse(plotdf[x_var], plotdf[y_var], normalization='std')
            print(f"For skylight={sl_val}, NRMSE: {nrmse:.4f}, RMSE: {rmse:.4f}")

        if return_data:
            data.loc[data["skylight"] == sl_val, x_var] = plotdf.loc[:, x_var]

    
    # Configure legends - collect from first axis and place at figure level
    handles, labels = axs[0].get_legend_handles_labels()
    for i in range(len(axs)):
        if axs[i].get_legend() is not None:
            axs[i].get_legend().remove()
    filtered = [(h, l) for h, l in zip(handles, labels) if l not in (None, '', '_nolegend_')]
    handles = [h for h, _ in filtered]
    labels = [l for _, l in filtered]
    
    if show_figure_legend and (not hue_is_numeric) and len(handles) > 0:
        if using_external_axes:
            fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.02),
                       fontsize=20, ncol=min(5, len(labels)), frameon=False)
        else:
            fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.93, 0.5),
                       fontsize=20, title='Room Type', title_fontsize=20)
    
    # Re-apply asymptote legend after removal loop (legacy behavior)
    if asymptote_legend_handles is not None and len(axs) > 2:
        axs[2].legend(asymptote_legend_handles, asymptote_legend_labels,
                      loc='upper left', fontsize=20)
    
    # Add numeric colorbar only when requested
    if add_numeric_colorbar and hue_is_numeric and scatter_collection_for_cbar is not None:
        colorbar_orientation = str(colorbar_orientation).lower()
        if colorbar_orientation not in ("horizontal", "vertical"):
            raise ValueError("colorbar_orientation must be 'horizontal' or 'vertical'.")
        if colorbar_rect is None:
            if colorbar_orientation == "vertical":
                colorbar_rect = (0.94, 0.15, 0.02, 0.7)
            else:
                colorbar_rect = (0.20, -0.03, 0.60, 0.03)
        cbar_ax = fig.add_axes(tuple(colorbar_rect))
        cbar_norm = hue_norm if hue_norm is not None else Normalize()
        cbar_mappable = ScalarMappable(norm=cbar_norm, cmap=palette_numeric)
        cbar_mappable.set_array([])
        cbar = fig.colorbar(cbar_mappable, cax=cbar_ax, orientation=colorbar_orientation)
        cbar.set_label(hue, fontsize=20)
        cbar.ax.tick_params(labelsize=20)
    
    if figure_suptitle:
        fig.suptitle(f"{model_name}: Modeled vs LES Flux with Sign-group Fits", fontsize=20)
    
    if not using_external_axes:
        plt.tight_layout(rect=[0, 0, 0.92, 0.95])
    
    if return_data and return_params:
        return fig, axs, data[x_var], fitted_params
    if return_data:
        return fig, axs, data[x_var]
    if return_params:
        return fig, axs, fitted_params
    return fig, axs


In [ ]:
# Use the plotting function with turbulence model
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Ventilation Model",
    p0=[1.0, 0.00000001],
    bounds=([0.99999, 0], [1.00001, 0.0000001]),
    adjustData=False,
    show_assymptotes=True
)

In [ ]:
# Use the plotting function with turbulence model
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Ventilation Model",
    p0=[1.0, 0.00000001],
    bounds=([0.1, 0], [np.inf, 0.0000001]),
    adjustData=False,
    show_assymptotes=True
)

In [ ]:
# Use the plotting function with turbulence model
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Ventilation Model",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True
)

In [ ]:
x_var2 = "rms-mass_flux-Norm"
q_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2].mean().to_dict()
q_rms_global = data[x_var2].mean()
print(f"Global mean {x_var2}: {q_rms_global:.4f}")
for sl in [0, 1]:
    for s in [1, -1]:
        label = "Skylight" if sl == 1 else "Window"
        direction = "In" if s == 1 else "Out"
        print(f"{label}-{direction} mean {x_var2}: {q_rms_means.get((sl, s), np.nan):.4f}")

# Use subgroup-specific initialization for the second fit parameter
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Ventilation Model",
    p0=[1.0, q_rms_global],
    bounds=([0.1, q_rms_global-0.00001], [np.inf, q_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=q_rms_means,
    group_param2_half_width=0.00001,
 )

In [ ]:
def ventilationReDecomp_qMeasured(X, a, u_rms):
    u_model, rms_scaling = X
    u_rms *= rms_scaling
    return pyafn.ventilationReDecomp_q(u_model, a, u_rms)

plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2,
    hue=x_var2,
    style="slAll",
    model_func=ventilationReDecomp_qMeasured,
    model_name="Turbulence-Enhanced q",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=False,
    show_assymptotes=False,
    add_numeric_colorbar=True
 )

In [ ]:
# Use the plotting function with turbulence model
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="Ventilation Model",
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True
)

In [ ]:
x_var2 = "p_rms-noInt-Norm"
p_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2].mean().to_dict()
p_rms_global = data[x_var2].mean()
print(f"Global mean {x_var2}: {p_rms_global:.4f}")
for sl in [0, 1]:
    for s in [1, -1]:
        label = "Skylight" if sl == 1 else "Window"
        direction = "In" if s == 1 else "Out"
        print(f"{label}-{direction} mean {x_var2}: {p_rms_means.get((sl, s), np.nan):.4f}")

# Use subgroup-specific initialization for the second fit parameter
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="Ventilation Model",
    p0=[1.0, p_rms_global],
    bounds=([0.1, p_rms_global-0.00001], [np.inf, p_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=p_rms_means,
    group_param2_half_width=0.00001,
 )

In [ ]:
def ventilationReDecomp_pMeasured(X, a, p_rms):
    u_model, rms_scaling = X
    p_rms *= rms_scaling
    return pyafn.ventilationReDecomp_p(u_model, a, p_rms)

# Combined paper-ready q+p figure (rows: q fit, q measured, p fit, p measured; cols: Window/Skylight)
fig_p, axs_p = plt.subplots(4, 2, figsize=(13, 22), dpi=160, sharex=True, sharey=True)

# Row 1: baseline q model (categorical hue)
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$R_Q(q)$ Fit",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    axes=axs_p[0, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Row 2: subgroup-mean initialized q model (categorical hue)
x_var2_q = "rms-mass_flux-Norm"
q_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2_q].mean().to_dict()
q_rms_global = data[x_var2_q].mean()
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$R_Q(q)$ Bulk Measurement",
    p0=[1.0, q_rms_global],
    bounds=([0.1, q_rms_global-0.00001], [np.inf, q_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=q_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_p[1, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Row 3: baseline p model (categorical hue)
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$R_Q(p)$ Fit",
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    axes=axs_p[2, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Row 4: subgroup-mean initialized p model (categorical hue)
x_var2_p = "p_rms-noInt-Norm"
p_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2_p].mean().to_dict()
p_rms_global = data[x_var2_p].mean()
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$R_Q(p)$ Bulk Measurement",
    p0=[1.0, p_rms_global],
    bounds=([0.1, p_rms_global-0.00001], [np.inf, p_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=p_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_p[3, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Paper-ready labels and layout
col_titles_p = ["Window", "Skylight"]
for j, t in enumerate(col_titles_p):
    axs_p[0, j].set_title(t, fontsize=24)

row_labels_p = ["$R_Q(q)$ Fit", "$R_Q(q)$ Measured", "$R_Q(p)$ Fit", "$R_Q(p)$ Measured"]
for i, row_label in enumerate(row_labels_p):
    axs_p[i, 0].set_ylabel(row_label, fontsize=24)

for ax in axs_p.flatten():
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)

fig_p.supxlabel("Modeled Flux Velocity", fontsize=24, y=0.04)
fig_p.supylabel("LES Flux Velocity", fontsize=24)
fig_p.suptitle("Ventilation Model Fits (q and p): Window vs Skylight", fontsize=24)
fig_p.subplots_adjust(left=0.13, right=0.93, top=0.94, bottom=0.08, wspace=0.13, hspace=0.12)

In [ ]:
data["Cp"] = data["p_rms-noInt-Norm"] / (rho / 2)

# Combined point-wise figure (rows: q and p, cols: Window/Skylight)
fig_pointwise, axs_pointwise = plt.subplots(2, 2, figsize=(13, 12), dpi=160, sharex=True, sharey=True)

# Point-wise q (top row)
q_hue_norm = plt.Normalize(vmin=0, vmax=0.25)
data_q_pointwise = data.copy()
plot_ventilation_model_fit(
    data=data_q_pointwise,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_q,
    hue=x_var2_q,
    style="slAll",
    model_func=ventilationReDecomp_qMeasured,
    model_name="$R_Q(q)$ Point-Wise Measurement",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=True,
    axes=axs_pointwise[0, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=True,
    hue_norm=q_hue_norm,
    palette_numeric="plasma",
    figure_suptitle=False,
    colorbar_rect=[0.92, 0.56, 0.015, 0.30],
    colorbar_orientation="vertical",
)

# Point-wise p (bottom row)
x_var2_p = "p_rms-noInt-Norm"
p_hue_norm = plt.Normalize(vmin=0, vmax=0.25)

data_p_pointwise = data.copy()
plot_ventilation_model_fit(
    data=data_p_pointwise,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_p,
    hue="Cp",
    style="slAll",
    model_func=ventilationReDecomp_pMeasured,
    model_name="$R_Q(p)$ Point-Wise Measurement",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=False,
    axes=axs_pointwise[1, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=True,
    hue_norm=p_hue_norm,
    palette_numeric="viridis",
    figure_suptitle=False,
    colorbar_rect=[0.92, 0.14, 0.015, 0.30],
    colorbar_orientation="vertical",
)

# Labels and layout
col_titles_pointwise = ["Window", "Skylight"]
for j, t in enumerate(col_titles_pointwise):
    axs_pointwise[0, j].set_title(t, fontsize=24)

axs_pointwise[0, 0].set_ylabel("$R_Q(q)$", fontsize=24)
axs_pointwise[1, 0].set_ylabel("$R_Q(p)$", fontsize=24)

for ax in axs_pointwise.flatten():
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)

fig_pointwise.supxlabel("Fluctuation-Corrected Modeled Flux Velocity", fontsize=24, y=0.05)
fig_pointwise.supylabel("LES Flux Velocity", fontsize=24)
fig_pointwise.suptitle("Point-Wise Ventilation Model Fits", fontsize=24)
fig_pointwise.subplots_adjust(left=0.10, right=0.90, top=0.90, bottom=0.12, wspace=0.2, hspace=0.1)

In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$R_Q(q)$ Fit",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

In [ ]:
plt.figure()
plt.scatter(xAdjusted, data[y_var], alpha=0.5, s=1)

In [ ]:
def aggregate_window_series_to_room(roomVentilationMI, window_series, out_col="xAdjusted_room", mode="abs_half_sum"):
    """
    window_series: pd.Series indexed like (run, windowKey), e.g. xAdjusted
    mode:
      - "abs_half_sum": sum(abs(q_i))/2  (ventilation magnitude style)
      - "sum":          signed sum(q_i)
      - "mean":         arithmetic mean
    """
    rv = roomVentilationMI.copy()
    rv[out_col] = np.nan

    window_key_cols = rv.columns[rv.columns.str.contains("windowKeys")].tolist()

    for (run, room), row in rv.iterrows():
        keys = row[window_key_cols].dropna().tolist()
        vals = []
        for k in keys:
            idx = (run, k)
            if idx in window_series.index:
                vals.append(window_series.loc[idx])

        if len(vals) == 0:
            continue

        vals = np.asarray(vals, dtype=float)
        if mode == "abs_half_sum":
            rv.loc[(run, room), out_col] = np.nansum(np.abs(vals)) / 2.0
        elif mode == "sum":
            rv.loc[(run, room), out_col] = np.nansum(vals)
        elif mode == "mean":
            rv.loc[(run, room), out_col] = np.nanmean(vals)
        else:
            raise ValueError(f"Unknown mode: {mode}")

    return rv

In [ ]:
minVent = roomVentilationMI[x_var].min()
maxVent = roomVentilationMI[x_var].max()
x_space = np.linspace(minVent, maxVent, 100)

windowModels = {}
for key, params in fittedParams.items():
    windowModels[key] = pyafn.ventilationReDecomp_q(x_space, *params)

roomModels = {}
roomResiduals = {}
for inOpening in ["Window", "Skylight"]:
    for outOpening in ["Window", "Skylight"]:
        roomModels[f"In {inOpening}; Out {outOpening}"] = (windowModels[(inOpening, 'Flow Entering')] + windowModels[(outOpening, 'Flow Exiting')]) / 2
        roomResiduals[f"In {inOpening}; Out {outOpening}"] = windowModels[(inOpening, 'Flow Entering')] - windowModels[(outOpening, 'Flow Exiting')]


In [ ]:
roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI, xAdjusted, out_col="xAdjusted_room", mode="abs_half_sum"
)

roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI_adj, xAdjusted, out_col="xAdjusted_room_residual", mode="mean"
)
roomVentilationMI_adj["xAdjusted_room_residual"] *= 2
roomVentilationMI_adj["xAdjusted_diff"] = roomVentilationMI_adj["xAdjusted_room"] - roomVentilationMI_adj[x_var]

plot_df = roomVentilationMI_adj.copy().sort_values("roomType")

fig, axs = plt.subplots(2, 2, figsize=(13, 11), dpi=160)

for i in range(2):
    ax = axs[0, i]
    ax.plot(x_space, x_space, 'r--', label='1:1 Line')

styles = ['-', '--', '-.', ':']
for i, (key, model) in enumerate(roomModels.items()):
    color = "k"#plt.get_cmap("tab20")(i % 20)
    axs[1, 0].plot(x_space, model, linestyle=styles[i % len(styles)], label=key, color=color, alpha=1, linewidth = 1)
    axs[1, 0].legend(fontsize=16)

for i, (key, residual) in enumerate(roomResiduals.items()):
    color = "k" #plt.get_cmap("tab20")(i % 20)
    axs[1, 1].plot(
        roomModels[key] - x_space,
        residual,
        linestyle=styles[i % len(styles)],
        label=key,
        color=color,
        alpha=1,
        linewidth = 1
    )
    axs[1, 1].set_xlim(-0.1, 0.1)
    axs[1, 1].set_ylim(-0.1, 0.1)

scatter_specs = [
    (x_var, "flux-Norm", "Baseline Model vs LES"),
    ("xAdjusted_room", "flux-Norm", "Aggregated Adjusted Model vs LES"),
    (x_var, "xAdjusted_room", "Baseline Model vs Aggregated Adjusted Model"),
    ("xAdjusted_diff", "xAdjusted_room_residual", "Aggregation Difference vs Residual"),
]

for ax, (xcol, ycol, title) in zip(axs.flatten(), scatter_specs):
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=ycol,
        hue="roomType",
        style = "slAll",
        palette="colorblind",
        alpha=0.6,
        s=10,
        ax=ax,
        legend=ax is axs[0, 0],
    )
 
    ax.set_title(title, fontsize=20)
    ax.set_xlabel(xcol, fontsize=18)
    ax.set_ylabel(ycol, fontsize=18)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=16)

handles, labels = axs[0, 0].get_legend_handles_labels()

for ax in axs.flatten():
    if ax.get_legend() is not None:
        ax.get_legend().remove()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.90, 0.5),
    fontsize=18,
    title="Room Type",
    title_fontsize=18,
    frameon=False,
)

fig.suptitle("Room-Level Aggregated Ventilation Comparisons", fontsize=24)
fig.subplots_adjust(left=0.10, right=0.87, top=0.90, bottom=0.10, wspace=0.28, hspace=0.30)

## ASHRAE Ventilation Rates

In [ ]:
windowASHRAE = pd.read_csv(f"{home_dir}/windowASHRAE.csv")

ashrae_lookup = windowASHRAE.set_index(["windowType", "roomA", "C"])["ventilationRate"]
data["ASHRAE"] = data.apply(
    lambda row: ashrae_lookup.get((row["windowType"], row["roomA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plot_df = data[data["slAll"] == False].copy()

fig, axs = plt.subplots(1, 3, figsize=(12, 4.8), dpi=160, sharey=True, layout="constrained")

scatter_specs = [
    ("ASHRAE", "ASHRAE Pressures",  "Modeled Flux Velocity"),
    (x_var, "LES Pressures", "Modeled Flux Velocity"),
    (xAdjusted, "LES Pressures", "Fluctuation-Adjusted Flux Velocity"),
]

for ax, (xcol, title, xlabel) in zip(axs, scatter_specs):
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=y_var,
        hue="roomType",
        # style="roomType",
        palette="colorblind",
        alpha=0.65,
        s=20,
        ax=ax,
        legend=ax is axs[0],
    )

    lim_min = np.nanmin([plot_df[x_var].min(), plot_df[y_var].min()])
    lim_max = np.nanmax([plot_df[x_var].max(), plot_df[y_var].max()])
    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        "r--",
        linewidth=1.2,
        alpha=0.7,
        label="1:1" if ax is axs[0] else None,
    )

    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("LES Flux Velocity", fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=14)
    # ax.set_xlim(-0.6, 0.6)
    # ax.set_ylim(-0.6, 0.6)


handles, labels = axs[0].get_legend_handles_labels()
for ax in axs:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

# fig.legend(
#     handles,
#     labels,
#     loc="center left",
#     bbox_to_anchor=(0.90, 0.5),
#     fontsize=12,
#     # title="Room A / Window Type",
#     title_fontsize=13,
#     frameon=False,
# )

fig.suptitle("Window-Level Ventilation Comparison", fontsize=20)
fig.subplots_adjust(left=0.08, right=0.86, top=0.83, bottom=0.18, wspace=0.30)

In [ ]:
data_WA_mean = data[data["slAll"] == False].groupby(["windowType", "roomA"])[y_var].mean().reset_index()

windowTypeOrder = data_WA_mean["windowType"].dropna().unique()
plt.figure()
sns.lineplot(data=data_WA_mean, x="roomA", y=y_var, hue="windowType", palette="tab10", hue_order=windowTypeOrder)
sns.lineplot(data=windowASHRAE, x="roomA", y="ventilationRate", hue="windowType", palette="tab10", linestyle='--', hue_order=windowTypeOrder)

In [ ]:
roomASHRAE = pd.read_csv(f"{home_dir}/roomASHRAE.csv")

ashrae_lookup = roomASHRAE.set_index(["roomType", "AofA", "C"])["ventilationRate"]
roomVentilationMI_adj["ASHRAE"] = roomVentilationMI_adj.apply(
    lambda row: ashrae_lookup.get((row["roomType"], row["AofA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plt.figure()
sns.scatterplot(data=roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False], x="ASHRAE", y=y_var, hue="AofA", palette="viridis")

In [ ]:
room_WA_mean = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False].groupby(["roomType", "AofA"])[y_var].mean().reset_index()

roomTypeOrder = room_WA_mean["roomType"].dropna().unique()
plt.figure()
sns.lineplot(data=room_WA_mean, x="AofA", y=y_var, hue="roomType", palette="tab10", hue_order=roomTypeOrder)
sns.lineplot(data=roomASHRAE, x="AofA", y="ventilationRate", hue="roomType", palette="tab10", linestyle='--', hue_order=roomTypeOrder)

In [ ]:
def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
import flowEmulationUtils as feUtils
flowStatsMI_directNetwork, roomVentilationMI_directNetwork = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
flowStatsMI_directNetwork["p-noInt-direct_optp0-q_model-Norm"] = flowStatsMI_directNetwork["p-noInt-direct_optp0-q_model"] / flowStatsMI_directNetwork["WS"]
roomVentilationMI_directNetwork["p-noInt-direct_optp0-q_model-Norm"] = roomVentilationMI_directNetwork["p-noInt-direct_optp0-q_model"] / roomVentilationMI_directNetwork["WS"]

flowStatsMI_directNetwork["flux-Norm"] = flowStatsMI_directNetwork["flux"] / flowStatsMI_directNetwork["WS"]

fig, axs = plt.subplots(1, 3, figsize=(12, 5), dpi=160, sharey=True, sharex=True)
# turn on grid
for ax in axs:
    ax.grid(True, alpha=0.3)
hue_order = ["corner", "cross", "dual", "single"]
style_order = ["xwindow", "zwindow", "skylight"]
sns.scatterplot(data=data, x=x_var, y=y_var, hue="roomType", style="openingType", palette="colorblind", ax=axs[0], size=0.5, alpha=0.7, hue_order=hue_order, style_order=style_order)
sns.scatterplot(data=flowStatsMI_directNetwork, x=xAdjusted, y=y_var, hue="roomType", style="openingType", palette="colorblind", ax=axs[1], size=0.5, alpha=0.7, hue_order=hue_order, style_order=style_order)
sns.scatterplot(data=flowStatsMI_directNetwork, x="p-noInt-direct_optp0-q_model-Norm", y=y_var, hue="roomType", style="openingType", palette="colorblind", ax=axs[2], size=0.5, alpha=0.7, hue_order=hue_order, style_order=style_order)

fig, axs = plt.subplots(1, 3, figsize=(12, 5), dpi=160, sharey=True, sharex=True)
# turn on grid
for ax in axs:
    ax.grid(True, alpha=0.3)
hue_order = ["corner", "cross", "dual", "single"]
style_order = ["xwindow", "zwindow", "skylight"]
sns.scatterplot(data=roomVentilationMI, x=x_var, y=y_var, hue="roomType", style="roomType", palette="colorblind", ax=axs[0], size=0.5, alpha=0.7, hue_order=hue_order, style_order=hue_order)
sns.scatterplot(data=roomVentilationMI_adj, x="xAdjusted_room", y=y_var, hue="roomType", style="roomType", palette="colorblind", ax=axs[1], size=0.5, alpha=0.7, hue_order=hue_order, style_order=hue_order)
sns.scatterplot(data=roomVentilationMI_directNetwork, x="p-noInt-direct_optp0-q_model-Norm", y=y_var, hue="roomType", style="roomType", palette="colorblind", ax=axs[2], size=0.5, alpha=0.7, hue_order=hue_order, style_order=hue_order)

In [ ]:
optParams = [6.121e-01, 6.467e-01, 5.840e-01, 6.464e-01, 1.271e-01, 1.010e-01, 9.888e-09, 1.309e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, optParams[4]]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, optParams[5]]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, optParams[6]]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, optParams[7]])
    }

def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
flowStatsMI_directNetwork2, roomVentilationMI_directNetwork2 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
flowStatsMI_directNetwork2["p-noInt-direct_optp0-q_model-Norm"] = flowStatsMI_directNetwork2["p-noInt-direct_optp0-q_model"] / flowStatsMI_directNetwork2["WS"]
roomVentilationMI_directNetwork2["p-noInt-direct_optp0-q_model-Norm"] = roomVentilationMI_directNetwork2["p-noInt-direct_optp0-q_model"] / roomVentilationMI_directNetwork2["WS"]

flowStatsMI_directNetwork2["flux-Norm"] = flowStatsMI_directNetwork2["flux"] / flowStatsMI_directNetwork2["WS"]

fig, axs = plt.subplots(1, 3, figsize=(12, 5), dpi=160, sharey=True, sharex=True)
# turn on grid
for ax in axs:
    ax.grid(True, alpha=0.3)
hue_order = ["corner", "cross", "dual", "single"]
style_order = ["xwindow", "zwindow", "skylight"]
sns.scatterplot(data=data, x=x_var, y=y_var, hue="roomType", style="openingType", palette="colorblind", ax=axs[0], size=0.5, alpha=0.7, hue_order=hue_order, style_order=style_order)
sns.scatterplot(data=flowStatsMI_directNetwork2, x=xAdjusted, y=y_var, hue="roomType", style="openingType", palette="colorblind", ax=axs[1], size=0.5, alpha=0.7, hue_order=hue_order, style_order=style_order)
sns.scatterplot(data=flowStatsMI_directNetwork2, x="p-noInt-direct_optp0-q_model-Norm", y=y_var, hue="roomType", style="openingType", palette="colorblind", ax=axs[2], size=0.5, alpha=0.7, hue_order=hue_order, style_order=style_order)

fig, axs = plt.subplots(1, 3, figsize=(12, 5), dpi=160, sharey=True, sharex=True)
# turn on grid
for ax in axs:
    ax.grid(True, alpha=0.3)
hue_order = ["corner", "cross", "dual", "single"]
style_order = ["xwindow", "zwindow", "skylight"]
sns.scatterplot(data=roomVentilationMI, x=x_var, y=y_var, hue="roomType", style="roomType", palette="colorblind", ax=axs[0], size=0.5, alpha=0.7, hue_order=hue_order, style_order=hue_order)
sns.scatterplot(data=roomVentilationMI_adj, x="xAdjusted_room", y=y_var, hue="roomType", style="roomType", palette="colorblind", ax=axs[1], size=0.5, alpha=0.7, hue_order=hue_order, style_order=hue_order)
sns.scatterplot(data=roomVentilationMI_directNetwork2, x="p-noInt-direct_optp0-q_model-Norm", y=y_var, hue="roomType", style="roomType", palette="colorblind", ax=axs[2], size=0.5, alpha=0.7, hue_order=hue_order, style_order=hue_order)

In [ ]:
optParams = [6.882e-01, 7.107e-01, 6.848e-01, 7.639e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, 0]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, 0]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, 0]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, 0])
    }

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

flowStatsMI["pRMS"] = flowStatsMI.apply(lambda x: [x["p_rms-noInt"], x["p_rms-noInt"]], axis=1)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
flowStatsMI_directNetwork3, roomVentilationMI_directNetwork3 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
flowStatsMI_directNetwork3["p-noInt-direct_optp0-q_model-Norm"] = flowStatsMI_directNetwork3["p-noInt-direct_optp0-q_model"] / flowStatsMI_directNetwork3["WS"]
roomVentilationMI_directNetwork3["p-noInt-direct_optp0-q_model-Norm"] = roomVentilationMI_directNetwork3["p-noInt-direct_optp0-q_model"] / roomVentilationMI_directNetwork3["WS"]

flowStatsMI_directNetwork3["flux-Norm"] = flowStatsMI_directNetwork3["flux"] / flowStatsMI_directNetwork3["WS"]

fig, axs = plt.subplots(1, 3, figsize=(12, 5), dpi=160, sharey=True, sharex=True)
# turn on grid
for ax in axs:
    ax.grid(True, alpha=0.3)
hue_order = ["corner", "cross", "dual", "single"]
style_order = ["xwindow", "zwindow", "skylight"]
sns.scatterplot(data=data, x=x_var, y=y_var, hue="roomType", style="openingType", palette="colorblind", ax=axs[0], size=0.5, alpha=0.7, hue_order=hue_order, style_order=style_order)
sns.scatterplot(data=flowStatsMI_directNetwork3, x=xAdjusted, y=y_var, hue="roomType", style="openingType", palette="colorblind", ax=axs[1], size=0.5, alpha=0.7, hue_order=hue_order, style_order=style_order)
sns.scatterplot(data=flowStatsMI_directNetwork3, x="p-noInt-direct_optp0-q_model-Norm", y=y_var, hue="roomType", style="openingType", palette="colorblind", ax=axs[2], size=0.5, alpha=0.7, hue_order=hue_order, style_order=style_order)

fig, axs = plt.subplots(1, 3, figsize=(12, 5), dpi=160, sharey=True, sharex=True)
# turn on grid
for ax in axs:
    ax.grid(True, alpha=0.3)
hue_order = ["corner", "cross", "dual", "single"]
style_order = ["xwindow", "zwindow", "skylight"]
sns.scatterplot(data=roomVentilationMI, x=x_var, y=y_var, hue="roomType", style="roomType", palette="colorblind", ax=axs[0], size=0.5, alpha=0.7, hue_order=hue_order, style_order=hue_order)
sns.scatterplot(data=roomVentilationMI_adj, x="xAdjusted_room", y=y_var, hue="roomType", style="roomType", palette="colorblind", ax=axs[1], size=0.5, alpha=0.7, hue_order=hue_order, style_order=hue_order)
sns.scatterplot(data=roomVentilationMI_directNetwork3, x="p-noInt-direct_optp0-q_model-Norm", y=y_var, hue="roomType", style="roomType", palette="colorblind", ax=axs[2], size=0.5, alpha=0.7, hue_order=hue_order, style_order=hue_order)